In [ ]:
LIST @audio_stage;

In [ ]:
-- Créer la table de métadonnées
CREATE OR REPLACE TABLE audio_metadata (
    file_name     VARCHAR,
    label         VARCHAR,        -- 'Asthma', 'COPD', 'Bronchial', 'Pneumonia', 'Healthy'
    duration_sec  FLOAT,
    sample_rate   INT,
    file_path     VARCHAR         -- chemin relatif dans le stage
);

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.files import SnowflakeFile
from scipy.io import wavfile
import io
import pandas as pd

# Fonction pour déduire la classe selon le nom en vrac
def extract_label_from_filename(filename):
    file_lower = filename.lower()
    if 'asthma' in file_lower or 'wheezing' in file_lower:
        return 'asthma'
    elif 'bronchial' in file_lower:
        return 'bronchial'
    elif 'copd' in file_lower:
        return 'copd'
    elif 'pneumonia' in file_lower:
        return 'pneumonia'
    elif 'healthy' in file_lower:
        return 'healthy'
    else:
        return 'unknown'

# 1. Connexion et Context
session = get_active_session()
session.sql("USE DATABASE TESSAN_AUDIO").collect()
session.sql("USE SCHEMA PUBLIC").collect()

# 2. Récupération de la liste (qui est en vrac à la racine du stage)
stage_files = session.sql("LIST @audio_stage").collect()

metadata_records = []

print(f"Début du traitement de {len(stage_files)} fichiers...")

for row in stage_files:
    file_path = row['name'] # ex: "audio_stage/P10AsthmaIE_49.wav"
    wav_file_name = file_path.split('/')[-1] # extraction propre du nom de fichier
    
    # 3. Extraction du label depuis le nom du fichier avec nos règles
    label = extract_label_from_filename(wav_file_name)
    
    try:
        # 4. Lecture saine via le channel binaire de Snowflake
        with SnowflakeFile.open(f"@{file_path}", 'rb') as f:
            sr, y = wavfile.read(io.BytesIO(f.read()))
            
        duration = len(y) / sr
        
        metadata_records.append({
            'FILENAME': wav_file_name,
            'LABEL': label,
            'DURATION': float(duration),
            'SAMPLE_RATE': int(sr),
            'STAGE_PATH': f"@{file_path}"
        })
    except Exception as e:
        print(f"Erreur de lecture sur {wav_file_name} : {e}")

# 5. Insertion en BATCH (Best Practice Snowflake)
if metadata_records:
    print("Création du DataFrame Snowpark...")
    pdf = pd.DataFrame(metadata_records)
    snow_df = session.create_dataframe(pdf)
    
    # Remplacer "append" par "overwrite" 
    # pour écraser les anciennes données corrompues !
    snow_df.write.mode("overwrite").save_as_table("audio_metadata")
    print(f"{len(metadata_records)} métadonnées insérées dans AUDIO_METADATA !")
else:
    print("Aucune donnée extraite.")

In [ ]:
# Q2 Exploration classe Snowflake
SELECT label, COUNT(*) as nb_echantillons
FROM audio_metadata
GROUP BY label
ORDER BY nb_echantillons DESC;

In [ ]:
# Q4 
from scipy.io import wavfile
from scipy import signal
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from snowflake.snowpark.files import SnowflakeFile
import io

session = get_active_session()

exemples = session.sql("""
    SELECT label, stage_path
    FROM audio_metadata
    WHERE label != 'unknown'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY label ORDER BY filename) = 1
""").to_pandas()

n = len(exemples)
fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))

# --- Paramètres normalisés fixes pour toutes les classes ---
DURATION_MAX   = 6.0      # toutes les formes d'onde affichées sur 10 secondes
AMPLITUDE_MIN  = -0.75      # amplitude normalisée entre -1 et 1
AMPLITUDE_MAX  =  0.75
FREQ_MIN       = 0         # fréquence min Hz
FREQ_MAX       = 4000      # fréquence max Hz (sons respiratoires utiles < 4 kHz)
DB_MIN         = -60       # dB min pour le spectrogramme
DB_MAX         =   0       # dB max (normalisé sur 0)

all_data = []

# --- Lecture de tous les fichiers d'abord ---
for i, row in exemples.iterrows():
    stage_path = row['STAGE_PATH']
    with SnowflakeFile.open(stage_path, 'rb') as f:
        sr, y = wavfile.read(io.BytesIO(f.read()))

    # Mono si stéréo
    if y.ndim == 2:
        y = y.mean(axis=1)

    # Normalisation amplitude entre -1 et 1
    if y.dtype != np.float32:
        y = y.astype(np.float32) / np.iinfo(y.dtype).max

    # Tronquer ou padder à DURATION_MAX secondes
    target_len = int(DURATION_MAX * sr)
    if len(y) > target_len:
        y = y[:target_len]
    else:
        y = np.pad(y, (0, target_len - len(y)))

    all_data.append((row['LABEL'], sr, y))

# --- Tracé avec échelles fixes ---
for i, (label, sr, y) in enumerate(all_data):

    # -- Forme d'onde --
    time = np.linspace(0, DURATION_MAX, len(y))
    axes[0, i].plot(time, y, linewidth=0.4, color='steelblue')
    axes[0, i].set_title(label, fontsize=12, fontweight='bold')
    axes[0, i].set_xlim([0, DURATION_MAX])
    axes[0, i].set_ylim([AMPLITUDE_MIN, AMPLITUDE_MAX])
    axes[0, i].set_xlabel('Temps (s)')
    if i == 0:
        axes[0, i].set_ylabel('Amplitude normalisée')
    else:
        axes[0, i].set_ylabel('')
        axes[0, i].set_yticklabels([])
    axes[0, i].axhline(0, color='gray', linewidth=0.3, linestyle='--')

    # -- Spectrogramme --
    f_spec, t_spec, Sxx = signal.spectrogram(
        y, sr,
        nperseg=1024,
        noverlap=512,
        window='hann'
    )

    # Convertir en dB et normaliser entre DB_MIN et DB_MAX
    Sxx_db = 10 * np.log10(Sxx + 1e-10)
    Sxx_db = Sxx_db - Sxx_db.max()          # ramène le max à 0 dB
    Sxx_db = np.clip(Sxx_db, DB_MIN, DB_MAX) # clip entre -60 et 0

    # Filtrer les fréquences utiles (0–4000 Hz)
    freq_mask = f_spec <= FREQ_MAX
    
    im = axes[1, i].pcolormesh(
        t_spec,
        f_spec[freq_mask],
        Sxx_db[freq_mask, :],
        shading='gouraud',
        cmap='magma',
        vmin=DB_MIN,
        vmax=DB_MAX       # échelle dB identique pour toutes les classes
    )
    axes[1, i].set_xlim([0, DURATION_MAX])
    axes[1, i].set_ylim([FREQ_MIN, FREQ_MAX])
    axes[1, i].set_xlabel('Temps (s)')
    if i == 0:
        axes[1, i].set_ylabel('Fréquence (Hz)')
    else:
        axes[1, i].set_ylabel('')
        axes[1, i].set_yticklabels([])

# Colorbar commune pour tous les spectrogrammes
cbar_ax = fig.add_axes([0.92, 0.08, 0.015, 0.35])
fig.colorbar(im, cax=cbar_ax, label='Intensité (dB)')

plt.suptitle("Formes d'onde et spectrogrammes — échelles normalisées", y=1.02, fontsize=14)
plt.tight_layout(rect=[0, 0, 0.91, 1])
plt.show()